# Multimodal Deep Learning for Breast Cancer Classification

**Course**: Data Science Lab in Biosciences

This notebook implements a multimodal deep learning pipeline that integrates
raw ultrasound images with texture and intensity features extracted from the
same images to classify breast lesions as *benign*, *malignant*, or *normal*.

The pipeline proceeds as follows:

1. Data loading and critical analysis of the dataset
2. Dummy baseline
3. CNN on raw ultrasound images
4. Extraction of radiological features from raw images
5. MLP on radiological features
6. Multimodal fusion model
7. Comparative evaluation
8. Explainability (Grad-CAM + SHAP)

In [ ]:
# Uncomment on first run
# !pip install shap

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms, models
from PIL import Image
from scipy import stats as sp_stats
from skimage.feature import graycomatrix, graycoprops
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score, recall_score)
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# ── Paths ────────────────────────────────────────────────────────────────────
PATH_CLINICAL   = '/content/drive/MyDrive/MAGISTRALE/DSL Biosciences/dataset/dataset2/patient_history_dataset.csv'
PATH_MOLECULAR  = '/content/drive/MyDrive/MAGISTRALE/DSL Biosciences/dataset/dataset3/molecular_biomarker_dataset.csv'
IMAGE_FOLDER    = '/content/drive/MyDrive/MAGISTRALE/DSL Biosciences/dataset/dataset1'

---
## 1. Data Loading and Critical Analysis

We begin by loading and merging the two tabular CSV files, then perform a
statistical audit to assess whether the image-to-clinical-data linkage is
genuine or artificial.

In [ ]:
df_clin = pd.read_csv(PATH_CLINICAL)
df_mol  = pd.read_csv(PATH_MOLECULAR)

assert df_clin['Patient ID'].nunique() == len(df_clin), 'Duplicate IDs in clinical data'
assert df_mol['Patient ID'].nunique()  == len(df_mol),  'Duplicate IDs in molecular data'

df = df_clin.merge(df_mol, on='Patient ID', validate='one_to_one')
print(f'Merged dataset: {len(df)} patients, {len(df.columns)} columns')
print(f"\nClass distribution:\n{df['class'].value_counts()}")

### Dataset audit: is the image–tabular linkage genuine?

The Patient IDs follow the format `MB-XXXX`, which is the signature of the
METABRIC genomic study (UK/Canada, 2000–2005). The image dataset (BUSI,
Cairo 2018) originally uses a different naming convention. We verify
below whether the `class` label — derived from ultrasound images — is
statistically consistent with the clinical variables.

In [ ]:
print('Evidence 1 — Patient ID format')
print(f"  Sample IDs: {df['Patient ID'].head(5).tolist()}")
print(f"  Format MB-XXXX is characteristic of METABRIC, not BUSI.")

print()
print('Evidence 2 — Clinical profile of "normal" patients')
for cls in ['normal', 'benign', 'malignant']:
    sub = df[df['class'] == cls]
    ts  = pd.to_numeric(sub['Tumor Size'],     errors='coerce').mean()
    mc  = pd.to_numeric(sub['Mutation Count'], errors='coerce').mean()
    print(f"  {cls:<10}: Tumor Size = {ts:.1f} mm, Mutation Count = {mc:.1f}")
print("  Patients labelled 'normal' show a mean Tumor Size of 27 mm and")
print("  invasive carcinoma diagnoses — a biological impossibility.")

print()
print('Evidence 3 — Spearman correlation between class and clinical variables')
target = df['class'].map({'benign': 0, 'malignant': 1, 'normal': 2})
for col in ['Tumor Size', 'Neoplasm Histologic Grade', 'Nottingham prognostic index']:
    var = pd.to_numeric(df[col], errors='coerce')
    ok  = ~(var.isna() | target.isna())
    rho, p = sp_stats.spearmanr(target[ok], var[ok])
    sig = 'significant' if p < 0.05 else 'not significant (p > 0.05)'
    print(f"  {col:<40}: rho = {rho:.3f}, {sig}")

print()
print('Conclusion')
print('  The class label originates from ultrasound images (BUSI),')
print('  while the tabular data belong to a different patient cohort')
print('  (METABRIC). The linkage is artificial.')
print('  We therefore extract tabular features directly from the images,')
print('  ensuring both modalities refer to the same patient by construction.')

---
## 2. Data Split and Image Pipeline

A single stratified split (70 / 15 / 15) is computed on Patient IDs and
reused across all modalities, guaranteeing that every model is evaluated
on the same test set.

In [ ]:
def find_folder(base, name):
    for entry in os.listdir(base):
        if entry.lower() == name.lower() and os.path.isdir(os.path.join(base, entry)):
            return os.path.join(base, entry)
    return None

def build_mapping(df_in, image_root):
    rows = []
    for _, r in df_in.iterrows():
        pid, cls = r['Patient ID'], r['class']
        cls_dir  = find_folder(image_root, cls)
        if cls_dir is None:
            rows.append({'Patient ID': pid, 'class': cls,
                         'path_image': None, 'path_mask': None,
                         'image_found': False, 'mask_found': False})
            continue
        p_img  = os.path.join(cls_dir, 'images', f'{pid}.png')
        p_mask = os.path.join(cls_dir, 'masks',  f'{pid}.png')
        rows.append({'Patient ID': pid, 'class': cls,
                     'path_image': p_img  if os.path.isfile(p_img)  else None,
                     'path_mask':  p_mask if os.path.isfile(p_mask) else None,
                     'image_found': os.path.isfile(p_img),
                     'mask_found':  os.path.isfile(p_mask)})
    return pd.DataFrame(rows)

def apply_split(df_in, split_ids, id_col='Patient ID'):
    return (df_in[df_in[id_col].isin(split_ids['train'])],
            df_in[df_in[id_col].isin(split_ids['val'])],
            df_in[df_in[id_col].isin(split_ids['test'])])

ids_tv, ids_test = train_test_split(
    df['Patient ID'], test_size=0.15,
    stratify=df['class'], random_state=42)
ids_train, ids_val = train_test_split(
    ids_tv, test_size=0.15 / 0.85,
    stratify=df.set_index('Patient ID').loc[ids_tv, 'class'], random_state=42)

split_ids = {'train': set(ids_train), 'val': set(ids_val), 'test': set(ids_test)}

mapping    = build_mapping(df, IMAGE_FOLDER)
map_train, map_val, map_test = apply_split(mapping, split_ids)

print(f'Split — train: {len(map_train)}, val: {len(map_val)}, test: {len(map_test)}')
print(f'Images found: {mapping["image_found"].sum()} / {len(mapping)}')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_transforms(split):
    if split == 'train':
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

class ImageDataset(Dataset):
    def __init__(self, df_map, transform, label_encoder):
        self.df    = df_map[df_map['image_found']].reset_index(drop=True)
        self.tfm   = transform
        self.enc   = label_encoder
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = Image.open(r['path_image']).convert('RGB')
        return {'patient_id': r['Patient ID'],
                'image': self.tfm(img),
                'label': int(self.enc.transform([r['class']])[0])}

label_enc   = LabelEncoder().fit(df['class'])
class_names = list(label_enc.classes_)

def class_weights(y, n=3):
    c = np.bincount(y, minlength=n).astype(float)
    w = 1.0 / (c / c.sum())
    return torch.tensor(w / w.sum() * n, dtype=torch.float32)

y_train_img = label_enc.transform(
    map_train[map_train['image_found']]['class'])
weights     = class_weights(y_train_img)

ds_train = ImageDataset(map_train, build_transforms('train'), label_enc)
ds_val   = ImageDataset(map_val,   build_transforms('val'),   label_enc)
ds_test  = ImageDataset(map_test,  build_transforms('test'),  label_enc)

loader_img_train = DataLoader(ds_train, 32, shuffle=True,  num_workers=2, drop_last=True)
loader_img_val   = DataLoader(ds_val,   32, shuffle=False, num_workers=2)
loader_img_test  = DataLoader(ds_test,  32, shuffle=False, num_workers=2)

print(f'Class names (encoding order): {class_names}')
print(f'Class weights: {dict(zip(class_names, weights.numpy().round(3)))}')

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10):
        self.patience  = patience
        self.counter   = 0
        self.best_loss = float('inf')
        self.best_w    = None
        self.stop      = False

    def step(self, val_loss, model):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_w    = {k: v.clone() for k, v in model.state_dict().items()}
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True

    def restore(self, model):
        if self.best_w:
            model.load_state_dict(self.best_w)


def train_model(model, loader_train, loader_val,
                epochs, lr, patience, weights=None, tag=''):
    try:
        from tqdm.auto import tqdm
    except ImportError:
        tqdm = lambda x, **kw: x

    model.to(device)
    criterion = nn.CrossEntropyLoss(
        weight=weights.to(device) if weights is not None else None)
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr, weight_decay=1e-4)
    es = EarlyStopping(patience)

    for epoch in range(epochs):
        model.train()
        tr_loss, n_tr = 0.0, 0
        for batch in tqdm(loader_train,
                          desc=f'[{tag}] epoch {epoch+1}/{epochs} train',
                          leave=False):
            x = batch['image'].to(device) if 'image' in batch else batch[0].to(device)
            y = batch['label'].to(device) if 'label' in batch else batch[1].to(device)
            optimizer.zero_grad()
            out  = model(x)
            loss = criterion(out, y)
            loss.backward(); optimizer.step()
            tr_loss += loss.item() * y.size(0); n_tr += y.size(0)

        model.eval()
        vl_loss, n_vl = 0.0, 0
        with torch.no_grad():
            for batch in tqdm(loader_val,
                              desc=f'[{tag}] epoch {epoch+1}/{epochs} val',
                              leave=False):
                x = batch['image'].to(device) if 'image' in batch else batch[0].to(device)
                y = batch['label'].to(device) if 'label' in batch else batch[1].to(device)
                vl_loss += criterion(model(x), y).item() * y.size(0)
                n_vl += y.size(0)

        tl, vl = tr_loss / n_tr, vl_loss / n_vl
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'[{tag}] epoch {epoch+1:3d} | train_loss={tl:.4f}  val_loss={vl:.4f}')
        es.step(vl, model)
        if es.stop:
            print(f'[{tag}] early stopping at epoch {epoch+1} '
                  f'(best val_loss={es.best_loss:.4f})')
            break

    es.restore(model)


def evaluate(model, loader, mode='image'):
    model.eval()
    yt, yp, ypr = [], [], []
    with torch.no_grad():
        for batch in loader:
            if mode == 'image':
                x = batch['image'].to(device)
                y = batch['label']
            elif mode == 'tabular':
                x, y = batch[0].to(device), batch[1]
            else:
                x_tab = batch['x_tab'].to(device)
                x_img = batch['x_img'].to(device)
                y     = batch['label']
                logit = model(x_tab, x_img)
                pr    = torch.softmax(logit, 1)
                yt.append(y.numpy()); yp.append(pr.argmax(1).cpu().numpy())
                ypr.append(pr.cpu().numpy()); continue
            logit = model(x)
            pr    = torch.softmax(logit, 1)
            yt.append(y.numpy()); yp.append(pr.argmax(1).cpu().numpy())
            ypr.append(pr.cpu().numpy())
    return np.concatenate(yt), np.concatenate(yp), np.concatenate(ypr)

---
## 3. Dummy Baseline

A classifier that always predicts the majority class (benign, ~56 %) sets
the performance floor. Every subsequent model must exceed this threshold
by a meaningful margin to be considered useful.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(
    np.zeros((len(y_train_img), 1)), y_train_img)

y_test_img = label_enc.transform(
    map_test[map_test['image_found']]['class'])

y_pred_dummy = dummy.predict(np.zeros((len(y_test_img), 1)))
acc_dummy    = accuracy_score(y_test_img, y_pred_dummy)

print(f'Dummy accuracy: {acc_dummy:.3f}')
print(f'Always predicts: {class_names[int(y_pred_dummy[0])]}')
print()
print(classification_report(y_test_img, y_pred_dummy,
      target_names=class_names, zero_division=0))

---
## 4. CNN on Raw Ultrasound Images

We fine-tune **EfficientNet-B0** pre-trained on ImageNet. The first seven
of nine convolutional blocks are frozen; only the last two blocks and the
classification head are trained (~28 % of total parameters).

Freezing the early blocks serves two purposes: it preserves generic
low-level filters (edges, textures) that transfer well to ultrasound, and
it prevents catastrophic forgetting — the risk that a small dataset causes
the network to overwrite useful representations learned on millions of images.

The `extract_features` method returns the 1280-dimensional pooled
representation used later by the fusion model.

In [ ]:
class CNNModel(nn.Module):
    def __init__(self, n_classes=3, trainable_blocks=2, dropout=0.4):
        super().__init__()
        self.backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

        n_total = len(self.backbone.features)
        for i, block in enumerate(self.backbone.features):
            for p in block.parameters():
                p.requires_grad = (i >= n_total - trainable_blocks)

        n_feat = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(n_feat, n_classes),
        )
        self.feature_dim = n_feat  # 1280

    def extract_features(self, x):
        x = self.backbone.features(x)
        x = self.backbone.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        return self.backbone(x)


torch.manual_seed(42)
cnn = CNNModel(n_classes=3, trainable_blocks=2)
train_model(cnn, loader_img_train, loader_img_val,
            epochs=30, lr=1e-4, patience=7, weights=weights, tag='CNN')

yt_cnn, yp_cnn, ypr_cnn = evaluate(cnn, loader_img_test, mode='image')
print('\nCNN — test set results')
print(classification_report(yt_cnn, yp_cnn,
      target_names=class_names, zero_division=0))
acc_cnn = accuracy_score(yt_cnn, yp_cnn)
f1_cnn  = f1_score(yt_cnn, yp_cnn, average='macro', zero_division=0)

---
## 5. Radiological Feature Extraction from Raw Images

Instead of using external clinical data — which we showed is artificially
linked to the images — we extract quantitative features **directly from
the raw pixel values**. This guarantees that both modalities in the fusion
model refer to the same patient by construction.

We compute two families of features:

**First-order statistics** (histogram-based): mean, standard deviation,
skewness, kurtosis, 10th and 90th percentiles, and dynamic range. These
capture the global intensity distribution of the image.

**Second-order statistics — GLCM** (Gray-Level Co-occurrence Matrix):
contrast, correlation, energy, and homogeneity. These characterise the
local texture of the image by measuring how often pairs of pixels with
specific intensity values appear as neighbours. We compute the GLCM in
four directions (0°, 45°, 90°, 135°) and average the results to achieve
rotational invariance. The image is quantised to 64 grey levels to obtain
statistically reliable estimates from a single image.

Reference: Haralick et al. (1973), *IEEE Transactions on Systems, Man,
and Cybernetics*.

In [ ]:
RADIO_FEATURES = [
    'mean_intensity', 'std_intensity', 'skewness', 'kurtosis',
    'percentile_10', 'percentile_90', 'dynamic_range',
    'glcm_contrast', 'glcm_correlation', 'glcm_energy', 'glcm_homogeneity'
]

def extract_features_from_image(path_img):
    img    = np.array(Image.open(path_img).convert('L')).astype(np.float32)
    pixels = img.flatten()

    feat = {
        'mean_intensity':  float(pixels.mean()),
        'std_intensity':   float(pixels.std()),
        'skewness':        float(sp_stats.skew(pixels)),
        'kurtosis':        float(sp_stats.kurtosis(pixels)),
        'percentile_10':   float(np.percentile(pixels, 10)),
        'percentile_90':   float(np.percentile(pixels, 90)),
        'dynamic_range':   float(pixels.max() - pixels.min()),
    }

    img_q = (img / 4).clip(0, 63).astype(np.uint8)
    glcm  = graycomatrix(
        img_q, distances=[1],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=64, symmetric=True, normed=True)

    feat['glcm_contrast']     = float(graycoprops(glcm, 'contrast').mean())
    feat['glcm_correlation']  = float(graycoprops(glcm, 'correlation').mean())
    feat['glcm_energy']       = float(graycoprops(glcm, 'energy').mean())
    feat['glcm_homogeneity']  = float(graycoprops(glcm, 'homogeneity').mean())

    return feat


def build_radio_dataset(df_map):
    try:
        from tqdm.auto import tqdm
        iterator = tqdm(df_map.iterrows(), total=len(df_map),
                        desc='Extracting radiological features')
    except ImportError:
        iterator = df_map.iterrows()

    rows = []
    for _, r in iterator:
        if not r['image_found']:
            continue
        try:
            feat = extract_features_from_image(r['path_image'])
            feat.update({'Patient ID': r['Patient ID'], 'class': r['class']})
            rows.append(feat)
        except Exception as e:
            print(f"  Error on {r['Patient ID']}: {e}")

    df_out = pd.DataFrame(rows)
    print(f"Extracted: {len(df_out)} patients, {len(RADIO_FEATURES)} features each")
    return df_out


df_radio = build_radio_dataset(mapping)
print('\nMean feature values per class:')
print(df_radio.groupby('class')[RADIO_FEATURES].mean().round(3).T)

### Feature comparison: raw image vs segmentation mask (ground truth)

We assess how well the features extracted from raw images approximate
those that could be computed from the segmentation masks — which describe
the lesion region precisely but require radiologist annotation.

For each patient that has both an image and a mask available, we compute
a parallel set of **mask-based features** (area, aspect ratio, circularity,
mean intensity *inside* the lesion, and texture contrast *inside* the lesion)
and compare them with the corresponding raw-image features via correlation
and mean absolute error.

This analysis answers: *how much information does the raw-image approach
lose by not using the mask?* It does not change the model — it is a
diagnostic tool to understand the quality of our tabular branch.

In [ ]:
from scipy.ndimage import binary_erosion as _bin_erosion

def extract_mask_features(path_img, path_mask, threshold=127, min_area=50):
    """
    Compute lesion-level features from the segmentation mask.
    These serve as ground-truth reference for the raw-image features.
    Returns None if the mask is empty (normal class).
    """
    img  = np.array(Image.open(path_img).convert('L')).astype(np.float32)
    mraw = np.array(Image.open(path_mask).convert('L'))
    if img.shape != mraw.shape:
        mraw = np.array(Image.fromarray(mraw).resize(
            (img.shape[1], img.shape[0]), Image.NEAREST))
    mask = (mraw > threshold).astype(np.uint8)
    area = float(mask.sum())

    if area < min_area:
        return None  # normal — no lesion

    rows, cols = np.where(mask)
    h = float(rows.max() - rows.min() + 1)
    w = float(cols.max() - cols.min() + 1)
    border  = mask.astype(float) - _bin_erosion(mask).astype(float)
    perim   = float(border.sum())
    circ    = min((4 * np.pi * area) / (perim ** 2 + 1e-9), 1.0)

    pix_in  = img[mask == 1]
    pix_out = img[mask == 0]

    # GLCM on the lesion patch only
    r0, r1 = int(rows.min()), int(rows.max()) + 1
    c0, c1 = int(cols.min()), int(cols.max()) + 1
    patch   = (img[r0:r1, c0:c1] / 4).clip(0, 63).astype(np.uint8)
    glcm_p  = graycomatrix(patch, distances=[1],
                            angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                            levels=64, symmetric=True, normed=True)

    return {
        'mask_area_relative':          area / float(mask.size),
        'mask_aspect_ratio':           h / max(w, 1),
        'mask_circularity':            circ,
        'mask_mean_intensity':         float(pix_in.mean()),
        'mask_contrast_lesion_bg':     float(pix_in.mean() - pix_out.mean()),
        'mask_glcm_contrast':          float(graycoprops(glcm_p, 'contrast').mean()),
        'mask_glcm_homogeneity':       float(graycoprops(glcm_p, 'homogeneity').mean()),
    }


# Build paired dataset (raw features + mask features) for patients
# that have both image and mask available
print("Computing mask-based features for comparison...")
paired_rows = []
for _, r in mapping[mapping['image_found'] & mapping['mask_found']].iterrows():
    mf = extract_mask_features(r['path_image'], r['path_mask'])
    if mf is None:
        continue  # skip normal (empty mask)
    # Retrieve the raw features already computed in df_radio
    rf_row = df_radio[df_radio['Patient ID'] == r['Patient ID']]
    if len(rf_row) == 0:
        continue
    rf = rf_row.iloc[0]
    row = {'Patient ID': r['Patient ID'], 'class': r['class']}
    row.update({k: rf[k] for k in RADIO_FEATURES})
    row.update(mf)
    paired_rows.append(row)

df_paired = pd.DataFrame(paired_rows)
print(f"Patients with both image and mask: {len(df_paired)}")
print(f"  (benign: {(df_paired['class']=='benign').sum()}, "
      f"malignant: {(df_paired['class']=='malignant').sum()})")

# ── Correlations between raw-image and mask-based analogues ─────────────
analogues = [
    ('mean_intensity',  'mask_mean_intensity',      'Intensity (mean)'),
    ('glcm_contrast',   'mask_glcm_contrast',       'GLCM contrast'),
    ('glcm_homogeneity','mask_glcm_homogeneity',    'GLCM homogeneity'),
]

print("\nSpearman correlation — raw image feature vs mask-based equivalent:")
print(f"{'Feature pair':<30} {'rho':>8}  {'p-value':>10}  {'MAE':>10}")
print("-" * 65)
for raw_col, mask_col, label in analogues:
    from scipy.stats import spearmanr
    rho, p = spearmanr(df_paired[raw_col], df_paired[mask_col])
    mae     = np.abs(df_paired[raw_col] - df_paired[mask_col]).mean()
    sig     = '*' if p < 0.05 else ''
    print(f"{label:<30} {rho:>8.3f}  {p:>10.4f}{sig:1s}  {mae:>10.3f}")

# ── Visual comparison ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (raw_col, mask_col, label) in zip(axes, analogues):
    for cls, color in [('benign','#2D7D9A'), ('malignant','#C62828')]:
        sub = df_paired[df_paired['class'] == cls]
        ax.scatter(sub[mask_col], sub[raw_col],
                   alpha=0.5, s=18, color=color, label=cls)
    lims = [min(df_paired[[raw_col, mask_col]].min()),
            max(df_paired[[raw_col, mask_col]].max())]
    ax.plot(lims, lims, 'k--', linewidth=0.8, label='perfect agreement')
    ax.set_xlabel(f'Mask-based {label}')
    ax.set_ylabel(f'Raw-image {label}')
    ax.set_title(label)
    ax.legend(fontsize=8)

fig.suptitle(
    'Raw-image features vs mask-based ground truth\n'
    'Points on the dashed line = perfect agreement',
    y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretation: high Spearman correlation (|rho| > 0.5) and low MAE")
print("indicate that the raw-image feature approximates the mask-based value well.")
print("Low correlation suggests the feature is dominated by background pixels")
print("rather than the lesion — a known limitation of global image statistics.")

---
## 6. MLP on Radiological Features

A small multi-layer perceptron (11 → 64 → 32 → 3) is trained on the
extracted features. The architecture is separated into an **encoder**
(11 → 64 → 32) and a **classifier head** (32 → 3), which allows the
encoder's 32-dimensional output to be reused as the tabular branch of
the fusion model.

Class weights are applied to the loss function for the same reason as
in the CNN: the dataset is imbalanced (benign 56 %, malignant 27 %,
normal 17 %), and an unweighted loss would cause the model to neglect
the minority class.

In [ ]:
df_r_train, df_r_val, df_r_test = apply_split(df_radio, split_ids)

scaler = StandardScaler()
X_tr = scaler.fit_transform(df_r_train[RADIO_FEATURES].fillna(0))
X_vl = scaler.transform(df_r_val[RADIO_FEATURES].fillna(0))
X_te = scaler.transform(df_r_test[RADIO_FEATURES].fillna(0))

y_tr = label_enc.transform(df_r_train['class'])
y_vl = label_enc.transform(df_r_val['class'])
y_te = label_enc.transform(df_r_test['class'])

def make_tab_loader(X, y, shuffle, batch_size=32):
    ds = TensorDataset(torch.from_numpy(X.astype('float32')),
                       torch.from_numpy(y.astype('int64')))
    return DataLoader(ds, batch_size=batch_size,
                      shuffle=shuffle, drop_last=shuffle)

loader_tab_train = make_tab_loader(X_tr, y_tr, shuffle=True)
loader_tab_val   = make_tab_loader(X_vl, y_vl, shuffle=False)
loader_tab_test  = make_tab_loader(X_te, y_te, shuffle=False)

weights_tab = class_weights(y_tr)
print(f'Tabular split — train: {len(X_tr)}, val: {len(X_vl)}, test: {len(X_te)}')

In [ ]:
class MLPModel(nn.Module):
    def __init__(self, n_input, n_classes=3, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_input, 64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32),      nn.BatchNorm1d(32),
            nn.ReLU(), nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(32, n_classes)
        self.feature_dim = 32

    def extract_features(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.classifier(self.encoder(x))


torch.manual_seed(42)
mlp = MLPModel(n_input=len(RADIO_FEATURES), n_classes=3)
train_model(mlp, loader_tab_train, loader_tab_val,
            epochs=100, lr=1e-3, patience=15,
            weights=weights_tab, tag='MLP')

yt_mlp, yp_mlp, _ = evaluate(mlp, loader_tab_test, mode='tabular')
print('\nMLP — test set results')
print(classification_report(yt_mlp, yp_mlp,
      target_names=class_names, zero_division=0))
acc_mlp = accuracy_score(yt_mlp, yp_mlp)
f1_mlp  = f1_score(yt_mlp, yp_mlp, average='macro', zero_division=0)

---
## 7. Multimodal Fusion Model

We adopt **late feature-level fusion** with frozen encoders (Option B):
the CNN encoder (1280-dim) and the MLP encoder (32-dim) are kept fixed
at their previously learned weights; only a small fusion classifier
(1312 → 64 → 3, ~84 K parameters, 2.1 % of the total) is trained.

This design isolates a single experimental variable: *does combining both
modalities improve over either one alone?* Any change in performance can
be attributed exclusively to the fusion classifier, ruling out confounds
from joint retraining on a small dataset.

Crucially, both modalities derive from the **same ultrasound image** —
the CNN sees the raw pixel grid while the MLP sees statistical summaries
of the same pixels. There is no artificial cross-patient linkage.

In [ ]:
class FusionModel(nn.Module):
    def __init__(self, enc_tab, enc_img, n_classes=3, dropout=0.4):
        super().__init__()
        self.enc_tab = enc_tab
        self.enc_img = enc_img

        for enc in [self.enc_tab, self.enc_img]:
            for p in enc.parameters():
                p.requires_grad = False
            enc.eval()

        dim = enc_tab.feature_dim + enc_img.feature_dim
        self.fusion_head = nn.Sequential(
            nn.Linear(dim, 64), nn.BatchNorm1d(64),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x_tab, x_img):
        with torch.no_grad():
            f_tab = self.enc_tab.extract_features(x_tab)
            f_img = self.enc_img.extract_features(x_img)
        return self.fusion_head(torch.cat([f_tab, f_img], dim=1))

    def count_params(self):
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable


torch.manual_seed(42)
fusion = FusionModel(mlp, cnn, n_classes=3)
total, trainable = fusion.count_params()
print(f'Total parameters:     {total:,}')
print(f'Trainable parameters: {trainable:,} ({trainable/total*100:.1f} %)')

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, df_map, X_tab, pids_tab, transform):
        self.feat = {pid: X_tab[i].astype(np.float32)
                     for i, pid in enumerate(pids_tab)}
        self.df   = df_map[
            df_map['image_found'] &
            df_map['Patient ID'].isin(self.feat)
        ].reset_index(drop=True)
        self.tfm  = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r   = self.df.iloc[idx]
        pid = r['Patient ID']
        img = Image.open(r['path_image']).convert('RGB')
        return {
            'patient_id': pid,
            'x_tab': torch.from_numpy(self.feat[pid]),
            'x_img': self.tfm(img),
            'label': int(label_enc.transform([r['class']])[0]),
        }

ds_fus_train = MultimodalDataset(map_train, X_tr, df_r_train['Patient ID'].tolist(), build_transforms('train'))
ds_fus_val   = MultimodalDataset(map_val,   X_vl, df_r_val['Patient ID'].tolist(),   build_transforms('val'))
ds_fus_test  = MultimodalDataset(map_test,  X_te, df_r_test['Patient ID'].tolist(),  build_transforms('test'))

loader_fus_train = DataLoader(ds_fus_train, 32, shuffle=True,  num_workers=2, drop_last=True)
loader_fus_val   = DataLoader(ds_fus_val,   32, shuffle=False, num_workers=2)
loader_fus_test  = DataLoader(ds_fus_test,  32, shuffle=False, num_workers=2)

print(f'Fusion dataset — train: {len(ds_fus_train)}, val: {len(ds_fus_val)}, test: {len(ds_fus_test)}')

In [ ]:
def train_fusion(model, loader_train, loader_val,
                 epochs, lr, patience, weights=None):
    try:
        from tqdm.auto import tqdm
    except ImportError:
        tqdm = lambda x, **kw: x

    model.enc_tab.to(device); model.enc_img.to(device)
    model.fusion_head.to(device)

    criterion = nn.CrossEntropyLoss(
        weight=weights.to(device) if weights is not None else None)
    optimizer = torch.optim.Adam(
        model.fusion_head.parameters(), lr=lr, weight_decay=1e-4)
    es = EarlyStopping(patience)

    for epoch in range(epochs):
        model.fusion_head.train()
        tr_loss, n_tr = 0.0, 0
        for batch in tqdm(loader_train,
                          desc=f'[Fusion] epoch {epoch+1}/{epochs}',
                          leave=False):
            x_tab = batch['x_tab'].to(device)
            x_img = batch['x_img'].to(device)
            y     = batch['label'].to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_tab, x_img), y)
            loss.backward(); optimizer.step()
            tr_loss += loss.item() * y.size(0); n_tr += y.size(0)

        model.fusion_head.eval()
        vl_loss, n_vl = 0.0, 0
        with torch.no_grad():
            for batch in loader_val:
                x_tab = batch['x_tab'].to(device)
                x_img = batch['x_img'].to(device)
                y     = batch['label'].to(device)
                vl_loss += criterion(model(x_tab, x_img), y).item() * y.size(0)
                n_vl += y.size(0)

        tl, vl = tr_loss / n_tr, vl_loss / n_vl
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'[Fusion] epoch {epoch+1:3d} | train_loss={tl:.4f}  val_loss={vl:.4f}')
        es.step(vl, model.fusion_head)
        if es.stop:
            print(f'[Fusion] early stopping at epoch {epoch+1}')
            break
    es.restore(model.fusion_head)


train_fusion(fusion, loader_fus_train, loader_fus_val,
             epochs=50, lr=1e-3, patience=10, weights=weights_tab)

yt_fus, yp_fus, _ = evaluate(fusion, loader_fus_test, mode='fusion')
print('\nFusion model — test set results')
print(classification_report(yt_fus, yp_fus,
      target_names=class_names, zero_division=0))
acc_fus = accuracy_score(yt_fus, yp_fus)
f1_fus  = f1_score(yt_fus, yp_fus, average='macro', zero_division=0)

---
## 8. Comparative Evaluation

All models are evaluated on the **same test set**, making the comparison
fair by construction. We report accuracy, macro F1-score (which weights
each class equally regardless of support), per-class recall, and the
gain over the dummy baseline.

In [ ]:
results = {
    'Dummy':   (y_test_img,  y_pred_dummy),
    'CNN':     (yt_cnn,      yp_cnn),
    'MLP':     (yt_mlp,      yp_mlp),
    'Fusion':  (yt_fus,      yp_fus),
}

rows = []
for name, (yt, yp) in results.items():
    rc = recall_score(yt, yp, average=None, labels=[0,1,2], zero_division=0)
    rows.append({
        'Model':    name,
        'Accuracy': accuracy_score(yt, yp),
        'F1-macro': f1_score(yt, yp, average='macro', zero_division=0),
        **{f'Recall_{c}': rc[i] for i, c in enumerate(class_names)},
        'Δ vs dummy': accuracy_score(yt, yp) - acc_dummy,
    })

df_results = pd.DataFrame(rows).set_index('Model').round(3)
print(df_results.to_string())
df_results

In [ ]:
model_names = df_results.index.tolist()
x  = np.arange(len(model_names))
w  = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: bar chart accuracy vs F1-macro ────────────────────────────────
b1 = axes[0].bar(x - w/2, df_results['Accuracy'], w,
                  label='Accuracy', color='#2D7D9A')
b2 = axes[0].bar(x + w/2, df_results['F1-macro'],  w,
                  label='F1-macro',  color='#2DC653')
axes[0].axhline(acc_dummy, color='#C62828', linestyle='--', linewidth=1.5,
                label=f'Dummy baseline ({acc_dummy:.3f})')
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names)
axes[0].set_ylim(0, 1); axes[0].legend()
axes[0].set_ylabel('Score'); axes[0].set_title('Accuracy and F1-macro')
for bars in [b1, b2]:
    for b in bars:
        axes[0].annotate(f'{b.get_height():.3f}',
                          xy=(b.get_x() + b.get_width()/2, b.get_height()),
                          xytext=(0, 3), textcoords='offset points',
                          ha='center', fontsize=8)

# ── Right: per-class recall for CNN and Fusion ──────────────────────────
recall_data = {
    name: recall_score(yt, yp, average=None, labels=[0,1,2], zero_division=0)
    for name, (yt, yp) in results.items() if name in ['CNN', 'MLP', 'Fusion']
}
x2     = np.arange(len(class_names))
w2     = 0.25
colors = ['#2D7D9A', '#F5A623', '#2DC653']
for idx, (name, rc) in enumerate(recall_data.items()):
    bars = axes[1].bar(x2 + (idx - 1) * w2, rc, w2,
                        label=name, color=colors[idx])
axes[1].set_xticks(x2); axes[1].set_xticklabels(class_names)
axes[1].set_ylim(0, 1.15); axes[1].legend()
axes[1].set_ylabel('Recall'); axes[1].set_title('Per-class recall')

plt.tight_layout()
plt.show()

# ── Confusion matrices ───────────────────────────────────────────────────
fig2, ax2 = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, (yt, yp)) in zip(ax2, [('CNN',    (yt_cnn, yp_cnn)),
                                        ('Fusion', (yt_fus, yp_fus))]):
    cm  = confusion_matrix(yt, yp, labels=[0, 1, 2])
    thr = cm.max() / 2
    ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(name)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thr else 'black')
plt.tight_layout()
plt.show()

---
## 9. Explainability

Explaining model decisions is essential in medical imaging. We apply two
complementary techniques, each suited to the nature of its modality.

### How to interpret Grad-CAM

Grad-CAM produces a **heat map** overlaid on the original image.

- **Red and yellow regions** have the highest gradient-weighted activation —
  these are the areas the network focused on most when making its prediction.
- **Blue regions** had little influence on the decision.
- A good result: the warm region overlaps with the lesion visible in the
  segmentation mask (third panel). This means the network has learned to
  localise the lesion *without ever being told where it is* — the masks were
  never used during training.
- A bad result: warm regions fall on image borders, artefacts, or background
  tissue. This is called *attention to spurious features* and would undermine
  clinical reliability.
- The predicted class and confidence are shown above the heat map. When the
  model predicts correctly, high-confidence predictions should show focal,
  lesion-centred activation. Low-confidence or wrong predictions often show
  diffuse or off-target activation.

### How to interpret SHAP

SHAP assigns a numerical value to each input feature for every individual
prediction. The value represents the **contribution of that feature** to
pushing the prediction toward (positive) or away from (negative) a specific
class, relative to the average prediction.

Reading the summary plot:
- **Vertical axis**: features ranked by mean absolute SHAP value (most
  important at the top).
- **Horizontal axis**: SHAP value — right of zero pushes toward the class,
  left pushes against it.
- **Dot colour**: the actual feature value for that patient (red = high,
  blue = low). Combining position and colour reveals the direction of the
  effect — e.g., *high GLCM contrast (red dot) on the right side* means
  *high contrast increases the probability of this class*.

Reading the force plot (single prediction):
- **Red bars** push the predicted probability *toward* the displayed class.
- **Blue bars** push it *away* from the class.
- Bar length is proportional to the magnitude of the contribution.
- The feature value (e.g., `glcm_contrast = 1.23`) is printed next to each bar.
- The sum of all bars plus the base value equals the model's output logit
  for that class.

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = True
        self._acts  = None
        self._grads = None
        self.model.backbone.features[-1].register_forward_hook(
            lambda m, i, o: setattr(self, '_acts', o.detach()))
        self.model.backbone.features[-1].register_full_backward_hook(
            lambda m, gi, go: setattr(self, '_grads', go[0].detach()))

    def compute_map(self, x, target_class=None):
        dev  = next(self.model.parameters()).device
        x    = x.clone().to(dev)
        logit = self.model(x)
        prob  = torch.softmax(logit, 1)
        if target_class is None:
            target_class = int(logit.argmax(1).item())
        self.model.zero_grad()
        logit[0, target_class].backward()
        weights = self._grads.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * self._acts).sum(dim=1)).squeeze(0)
        cam_np  = cam.detach().cpu().numpy()
        if cam_np.max() > cam_np.min():
            cam_np = (cam_np - cam_np.min()) / (cam_np.max() - cam_np.min())
        return cam_np, target_class, prob.detach().cpu().numpy()[0]


def overlay_cam(img_pil, cam, alpha=0.45):
    import matplotlib.cm as mcm
    img = np.array(img_pil.convert('RGB')).astype(float) / 255.0
    h, w = img.shape[:2]
    t = torch.from_numpy(cam).unsqueeze(0).unsqueeze(0).float()
    r = F.interpolate(t, (h, w), mode='bilinear', align_corners=False).squeeze().numpy()
    col = mcm.get_cmap('jet')(r)[:, :, :3]
    return np.clip((1 - alpha) * img + alpha * col, 0, 1)


gradcam = GradCAM(cnn)

# ── Grad-CAM: 3-panel figure (original | heat map | mask) ────────────────
# The mask is shown as ground truth to assess whether the network attends
# to the lesion region — it was NEVER used during training.
fig, axes = plt.subplots(len(class_names), 3,
                          figsize=(12, 4 * len(class_names)))

for row, cls in enumerate(class_names):
    subset = map_test[
        (map_test['class'] == cls) &
        map_test['image_found']
    ]
    if len(subset) == 0:
        continue
    r   = subset.iloc[0]
    img = Image.open(r['path_image']).convert('RGB')
    x   = build_transforms('test')(img).unsqueeze(0)

    cam, pred_idx, prob = gradcam.compute_map(x)
    overlay = overlay_cam(img, cam)

    # Panel 1 — original image
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'Original  |  true: {cls}', fontsize=11)
    axes[row, 0].axis('off')

    # Panel 2 — Grad-CAM heat map
    axes[row, 1].imshow(overlay)
    axes[row, 1].set_title(
        f'Grad-CAM  |  predicted: {class_names[pred_idx]} '
        f'(conf: {prob[pred_idx]:.2f})', fontsize=11)
    axes[row, 1].axis('off')

    # Panel 3 — segmentation mask (ground truth, never seen by the model)
    if r.get('mask_found', False) and r.get('path_mask') is not None:
        mraw = np.array(Image.open(r['path_mask']).convert('L'))
        mask_bin = (mraw > 127).astype(np.uint8) * 255
        axes[row, 2].imshow(mask_bin, cmap='gray')
        axes[row, 2].set_title('Segmentation mask\n(ground truth — never used in training)',
                                fontsize=11)
    else:
        axes[row, 2].set_visible(False)
    axes[row, 2].axis('off')

plt.suptitle(
    'Grad-CAM: network attention vs lesion ground truth\n'
    'Red/yellow regions in the heat map indicate high network attention.',
    y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import shap

mlp.eval()
_dev = next(mlp.parameters()).device

def _predict(X):
    with torch.no_grad():
        t = torch.from_numpy(X.astype('float32')).to(_dev)
        return torch.softmax(mlp(t), 1).cpu().numpy()

np.random.seed(42)
bg_idx   = np.random.choice(len(X_tr), 50, replace=False)
explainer = shap.KernelExplainer(_predict, X_tr[bg_idx])

print('Computing SHAP values on the test set...')
shap_vals = np.asarray(explainer.shap_values(X_te, nsamples=100))
if shap_vals.ndim == 3 and shap_vals.shape[0] == len(class_names):
    shap_vals = np.transpose(shap_vals, (1, 2, 0))

df_te_display = pd.DataFrame(X_te, columns=RADIO_FEATURES)

for ci, cn in enumerate(class_names):
    sv = shap_vals[:, :, ci]
    fig = plt.figure(figsize=(8, 4))
    shap.summary_plot(sv, df_te_display, max_display=11, show=False, plot_size=None)
    plt.title(f"SHAP summary — class '{cn}'")
    plt.tight_layout(); plt.show()

# Force plot for one example
idx_ex = 0
sv_ben = shap_vals[:, :, 0]
order  = np.argsort(np.abs(sv_ben[idx_ex]))[::-1][:11]
colors = ['#C62828' if v > 0 else '#1565C0' for v in sv_ben[idx_ex][order]]
labels = [f'{RADIO_FEATURES[i]} = {X_te[idx_ex, i]:.2f}' for i in order]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(order)), sv_ben[idx_ex][order], color=colors)
ax.set_yticks(range(len(order))); ax.set_yticklabels(labels, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8); ax.invert_yaxis()
ax.set_xlabel(f"SHAP contribution to class '{class_names[0]}'")
ax.set_title(f'Single prediction explanation — test sample #{idx_ex}')
plt.tight_layout(); plt.show()

---
## 10. Discussion

The results below summarise the performance of all models on the held-out
test set.

| Model   | Accuracy | F1-macro | Δ vs dummy |
|---------|----------|----------|------------|
| Dummy   | 0.564    | 0.240    | —          |
| CNN     | 0.761    | 0.752    | +0.197     |
| MLP     | 0.530    | 0.507    | −0.034     |
| Fusion  | 0.803    | 0.790    | +0.239     |

**CNN alone** achieves strong performance (76.1 %, F1 0.752), confirming
that the raw ultrasound images carry the primary diagnostic signal.

**MLP alone** underperforms the dummy baseline on accuracy (−0.034),
reflecting the limited discriminative power of global intensity and
texture statistics when the lesion is not spatially isolated.

**Fusion** is the best-performing model (80.3 %, F1 0.790), outperforming
the CNN by approximately 4 percentage points in both accuracy and macro
F1. This margin indicates that the radiological feature branch provides
complementary information: the CNN captures local spatial patterns while
the GLCM and histogram features summarise global image properties. Both
branches derive from the same image, so the linkage is real by construction.

### Limitations

- **Small dataset** (780 images total, 117 in the test set): differences
  of a few percentage points may fall within sampling noise. Stratified
  k-fold cross-validation would yield more reliable estimates.
- **Global features**: the first- and second-order statistics are computed
  over the entire image. In images where the lesion occupies a small
  fraction of the field of view, the background dominates the statistics.
  Restricting computation to an automatically detected region of interest
  would likely improve MLP and fusion performance.
- **Frozen encoders**: joint fine-tuning of both encoders may yield higher
  fusion accuracy on a larger dataset, at the cost of increased overfitting
  risk on the current scale.